# Лабораторная работа 9. Спектральные и кепстральные признаки аудио

**Цель.** Рассмотреть спектральное представление аудиосигнала, выполнить low-pass фильтрацию через FFT и получить MFCC-признаки для дальнейшего машинного обучения.

В референсах аудиофайлы не хранятся в репозитории. Поэтому данные не добавляются автоматически: для полного запуска положите `Alesis-S4-Plus-Clean-Gtr-C4.wav` и `1980s-Casio-Violin-C5.wav` в папку `data/`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import rfft, rfftfreq, irfft

np.random.seed(42)

DATA_DIR = Path('data')
AUDIO_MAIN = DATA_DIR / 'Alesis-S4-Plus-Clean-Gtr-C4.wav'
AUDIO_SECOND = DATA_DIR / '1980s-Casio-Violin-C5.wav'
DATA_DIR.mkdir(exist_ok=True)

if not AUDIO_MAIN.exists() or not AUDIO_SECOND.exists():
    print('Аудиофайлы не найдены. Добавлять отсутствующие данные не будем; положите WAV-файлы в папку data/ для полного запуска.')


## FFT-фильтрация

Ниже сигнал переводится в частотную область, частоты выше 2000 Гц зануляются, затем выполняется обратное преобразование. Блок пропускается, если основного WAV-файла нет.


In [ ]:
if AUDIO_MAIN.exists():
    samplerate, signal = wavfile.read(AUDIO_MAIN)
    if signal.ndim > 1:
        signal = signal[:, 0]
    signal = signal.astype(float)
    time = np.arange(len(signal)) / samplerate

    spectrum = rfft(signal)
    freqs = rfftfreq(len(signal), 1 / samplerate)
    filtered_spectrum = spectrum.copy()
    filtered_spectrum[freqs > 2000] = 0
    filtered_signal = irfft(filtered_spectrum, n=len(signal))

    print('Sampling rate:', samplerate)
    print('Samples:', len(signal))
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(time[:5000], signal[:5000])
    plt.title('Исходный сигнал')
    plt.subplot(2, 1, 2)
    plt.plot(time[:5000], filtered_signal[:5000])
    plt.title('Сигнал после low-pass FFT')
    plt.tight_layout()
    plt.show()
else:
    print('Пропуск FFT-блока: файл data/Alesis-S4-Plus-Clean-Gtr-C4.wav отсутствует.')


## MFCC-признаки

MFCC компактно описывают форму спектра и часто используются как признаки для классификации аудио. Расчёт выполняется через `librosa`, если библиотека установлена и файл присутствует.


In [ ]:
if AUDIO_MAIN.exists():
    try:
        import librosa
        import librosa.display
    except ImportError:
        print('Библиотека librosa не установлена; установите зависимости из requirements.txt для расчёта MFCC.')
    else:
        y, sr = librosa.load(AUDIO_MAIN, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        print('MFCC shape:', mfcc.shape)
        plt.figure(figsize=(10, 5))
        librosa.display.specshow(mfcc, x_axis='time', sr=sr)
        plt.colorbar(label='MFCC')
        plt.title('13 MFCC коэффициентов')
        plt.tight_layout()
        plt.show()
else:
    print('Пропуск MFCC-блока: основной аудиофайл отсутствует.')


In [ ]:
if AUDIO_SECOND.exists():
    try:
        from python_speech_features import mfcc, logfbank
    except ImportError:
        print('Библиотека python_speech_features не установлена; установите зависимости из requirements.txt.')
    else:
        rate, sig = wavfile.read(AUDIO_SECOND)
        mfcc_feat = mfcc(sig, rate)
        fbank_feat = logfbank(sig, rate)
        print('MFCC:', mfcc_feat.shape)
        print('Filter bank:', fbank_feat.shape)
else:
    print('Пропуск второго аудио-блока: файл data/1980s-Casio-Violin-C5.wav отсутствует.')


## Вывод

FFT позволяет управлять сигналом через частотные компоненты, а MFCC дают компактное признаковое описание спектра. При отсутствии исходных аудиофайлов ноутбук остаётся запускаемым и явно показывает, какие данные нужны для полного расчёта.
